# VeriYönetim — Sorgu Planlayıcı İnce Ayarı (QLoRA)

Qwen2.5-Coder-7B-Instruct modelini, Türkçe iş sorusunu **JSON sorgu planına** çevirmek
üzere ince ayarlar.

**Neden QLoRA?** 7 milyar parametrenin tamamını eğitmek 100 GB üzeri ekran kartı belleği
ister; Kaggle'da 16 GB var. QLoRA iki şeyi birden yapar:

* **Q (quantized)** — taban model 4 bit'e sıkıştırılıp *dondurulur*, ~4,5 GB yer kaplar
* **LoRA** — donmuş katmanların yanına küçük ek matrisler takılır, **yalnız onlar** eğitilir

Sonuç 15 GB'lık yeni bir model değil, birkaç yüz MB'lık bir **eklenti**. Eğitim bitince
taban modelle birleştirilip GGUF'a çevriliyor ve Ollama'ya kuruluyor.

**Ön koşul:** `train.jsonl` ve `eval.jsonl` dosyalarini Kaggle'a bir veri seti olarak
yükleyin (Add Data -> New Dataset) ve asagidaki `VERI_YOLU`nu ona gore duzeltin.
Dosyalar `tools/VeriYonetim.TrainingData` aracinin `build` komutundan cikar.

**Donanim:** Notebook -> Settings -> Accelerator = **GPU T4 x2** (tek GPU kullanilacak).


## 1. Kurulum

In [ ]:
%%capture
# Unsloth: QLoRA egitimini hizlandiran ve bellegi ciddi olcude dusuren sarmalayici.
# Kaggle imaji sik degistigi icin surumler genis birakildi; kurulum ~3 dakika surer.
!pip install -q -U "unsloth" "trl<0.16" peft accelerate bitsandbytes


In [ ]:
import os, json, torch
from unsloth import FastLanguageModel

# Bizim istem ~1200-1400 belirtec, plan ~60 belirtec. 2048 rahat yetiyor ve en uygunu:
# siniri buyutmek dogrudan egitim suresini uzatir.
MAX_SEQ = 2048

# Ollama'daki qwen2.5-coder:7b ile AYNI taban model. Farkli bir tabandan egitilseydi
# uretilen eklenti canlidaki modele takilamazdi.
TABAN_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct"

print("GPU:", torch.cuda.get_device_name(0))
print("Bellek:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


## 2. Modeli 4 bit yukle ve LoRA eklentisini tak

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = TABAN_MODEL,
    max_seq_length = MAX_SEQ,
    dtype = None,          # T4 -> float16, daha yeni kartlarda bfloat16
    load_in_4bit = True,   # QLoRA'nin "Q" tarafi
)

model = FastLanguageModel.get_peft_model(
    model,
    # r: eklenti matrislerinin "genisligi". Kucuk olursa ogrenme kapasitesi yetmez,
    # buyuk olursa ezberler. Sabit bir sablon dili ogrettigimiz icin 32 fazlasiyla yeter.
    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    # Dikkat katmanlarinin tamami + ileri besleme. Yalniz q/v secmek yaygin bir kisayol
    # ama BICIM (JSON) ogrenmede ileri besleme katmanlari da ise yariyor.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",  # bellek icin: ara sonuclar saklanmaz, yeniden hesaplanir
    random_state = 42,
)

model.print_trainable_parameters()


## 3. Veriyi yukle

Her satir bir egitim ornegi: `prompt` modelin gorecegi istem, `completion` uretmesi
beklenen plan. Istem, canlida calisan `QueryPromptBuilder` sinifinin **orneksiz**
bicimiyle uretildi - egitimdeki ve canlidaki istem birebir ayni olmali.

In [ ]:
VERI_YOLU = "/kaggle/input/veriyonetim-plan-egitim"   # kendi veri setinizin yolu

def oku(dosya):
    with open(os.path.join(VERI_YOLU, dosya), encoding="utf-8") as f:
        return [json.loads(satir) for satir in f if satir.strip()]

egitim = oku("train.jsonl")
degerlendirme = oku("eval.jsonl")

print(f"egitim: {len(egitim)}  degerlendirme: {len(degerlendirme)}")
print()
print("ornek soru  :", egitim[0]["question"])
print("beklenen plan:", egitim[0]["completion"])
print()

# Tarif dagilimi: bir soru tipi cok az temsil ediliyorsa model onu ogrenmez.
from collections import Counter
for tarif, adet in Counter(r["recipe"] for r in egitim).most_common():
    print(f"  {tarif:<24} {adet}")


## 4. Sohbet bicimine cevir

Ollama `/api/generate` cagrisinda istemi modelin kendi sohbet sablonuna yerlestirir.
Egitimi de ayni sablonla yapmaliyiz; yoksa model egitimde gordugunden farkli bir metin
gorur ve dogruluk duser.

In [ ]:
def bicimle(kayit):
    mesajlar = [
        {"role": "user",      "content": kayit["prompt"]},
        {"role": "assistant", "content": kayit["completion"]},
    ]
    return tokenizer.apply_chat_template(mesajlar, tokenize=False)

from datasets import Dataset
veri = Dataset.from_list([{"text": bicimle(k)} for k in egitim]).shuffle(seed=42)

print(veri[0]["text"][:500])
print("   ...")
print(veri[0]["text"][-250:])


In [ ]:
# Uzunluk denetimi: kesilen bir ornek, sonu olmayan bir plan ogretir - sessiz zehir.
ornekler = veri.select(range(min(500, len(veri))))
uzunluklar = [len(tokenizer(m["text"])["input_ids"]) for m in ornekler]

print(f"belirtec: ortalama {sum(uzunluklar)//len(uzunluklar)}, "
      f"en uzun {max(uzunluklar)}, sinir {MAX_SEQ}")
assert max(uzunluklar) < MAX_SEQ, "MAX_SEQ yetersiz - buyutun, yoksa ornekler kesilir"


## 5. Egitim

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

egitici = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = veri,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,     # etkin yigin = 16
        warmup_ratio = 0.05,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "/kaggle/working/ciktilar",
        report_to = "none",
    ),
)

# YALNIZ cevabi ogret. Bu satir olmadan model istemi de "uretmeyi" ogrenir; kaybin buyuk
# kismi istemden gelir ve asil onemli olan plan kismi gurultude kaybolur.
egitici = train_on_responses_only(
    egitici,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)


In [ ]:
# Kontrol: maskeleme dogru mu? Etiketlerde SADECE plan gorunmeli.
ornek = egitici.train_dataset[0]
etiketler = [t for t in ornek["labels"] if t != -100]
print("ogretilen kisim:", tokenizer.decode(etiketler))


In [ ]:
sonuc = egitici.train()

print()
print(f"Sure: {sonuc.metrics['train_runtime']/60:.1f} dakika")
print(f"Son kayip (loss): {sonuc.metrics['train_loss']:.4f}")


## 6. Hizli goz kontrolu

Degerlendirme kumesinden birkac soru. Asil olcum yerel makinede
`dotnet run -- evaluate` ile yapilacak (planlar ayni dogrulayicidan gecirilerek);
buradaki amac "model cikti bicimini tutturuyor mu" sorusunu erken gormek.

In [ ]:
FastLanguageModel.for_inference(model)

dogru_sayisi = 0
for kayit in degerlendirme[:12]:
    girdi = tokenizer.apply_chat_template(
        [{"role": "user", "content": kayit["prompt"]}],
        tokenize = True, add_generation_prompt = True, return_tensors = "pt",
    ).to("cuda")

    uretilen = model.generate(input_ids=girdi, max_new_tokens=256, do_sample=False)
    cevap = tokenizer.decode(uretilen[0][girdi.shape[1]:], skip_special_tokens=True).strip()

    dogru = cevap == kayit["completion"]
    dogru_sayisi += dogru
    print(("[TAM]  " if dogru else "[FARK] "), kayit["question"])
    if not dogru:
        print("        beklenen:", kayit["completion"])
        print("        uretilen:", cevap)

print(f"\nbirebir ayni: {dogru_sayisi}/12")


## 7. Kaydet

In [ ]:
# LoRA eklentisi: kucuk, tekrar egitmeden tasinabilir, taban modelle birlestirilir.
model.save_pretrained("/kaggle/working/lora")
tokenizer.save_pretrained("/kaggle/working/lora")
print("eklenti kaydedildi")


### GGUF'a cevir (Ollama icin)

Ollama GGUF bicimi okur. Eklenti taban modelle birlestirilip `q4_k_m` ile nicelenerek
tek dosyaya yaziliyor - canlida kullanilan `qwen2.5-coder:7b` ile ayni niceleme
seviyesi, yani hiz ve bellek davranisi degismiyor.

Bu hucre llama.cpp'yi derledigi icin 15-25 dakika surer. Takilirsa alternatif:
`model.save_pretrained_merged(...)` ile 16 bit kaydedip donusumu yerel makinede yapmak.

In [ ]:
model.save_pretrained_gguf(
    "/kaggle/working/veriyonetim-planlayici",
    tokenizer,
    quantization_method = "q4_k_m",
)


In [ ]:
# Ollama tanim dosyasi. Model adi "veriyonetim" ile BASLAMALI: sunucu bu onekten
# modelin ince ayarli oldugunu anliyor ve isteme few-shot ornekleri koymuyor
# (bkz. OllamaOptions.FineTunedPrefix).
modelfile = """FROM ./veriyonetim-planlayici.Q4_K_M.gguf

# Plan uretimi yaraticilik degil tekrarlanabilirlik ister.
PARAMETER temperature 0
PARAMETER num_ctx 4096
"""

with open("/kaggle/working/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

!ls -lh /kaggle/working/*.gguf /kaggle/working/Modelfile

print("""
Siradaki adimlar (yerel makinede):
  1. GGUF dosyasini ve Modelfile'i indir, ayni klasore koy
  2. ollama create veriyonetim-planlayici:7b -f Modelfile
  3. dotnet run --project tools/VeriYonetim.TrainingData -- evaluate \\
       --in data/samples.eval.jsonl --model veriyonetim-planlayici:7b
  4. Baz olcumle karsilastir
""")
